Check we have connection correct. To test this we call get a FHIR Conformance statement from the server

High-level purpose

This script:

- Loads FHIR server credentials and configuration from environment variables
- Builds a URL to a FHIR server’s metadata endpoint
- Sends an HTTP GET request to retrieve the server’s CapabilityStatement (what the FHIR server supports)

In short: it checks what a FHIR server can do.

In [ ]:
from dataclasses import asdict

import fhirclient
import requests
from fhirclient.models.fhirinstant import FHIRInstant
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv
load_dotenv()
import os

fhir_password = os.getenv("FHIR_PASSWORD")
fhir_username = os.getenv("FHIR_USERNAME")
#server = "https://gen-tie-test.nwgenomics.nhs.uk/dataplatform/cdr/fhir/r4/"
server = os.getenv("FHIR_SERVER")

api_url = server + "metadata"
print(api_url)
response = requests.get(api_url)
#response.json()

Find a patient. Simple search for a patient named wrexham.

The data used here is a combination of patient demographics from EHR systems (supplied with the orders) and [NHS England Personal Demographics Service - FHIR API](https://digital.nhs.uk/developer/api-catalogue/personal-demographics-service-fhir)

High-level purpose

This code:

- Searches a FHIR server for a Patient with the name "wrexham"
- Authenticates using HTTP Basic Auth
- Parses the FHIR search response (a Bundle)
- Extracts the first matching Patient ID

In short: it looks up a patient and retrieves their FHIR logical ID.

In [ ]:


api_url = server + "Patient?name=wrexham"
print(api_url)
response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
patientJSON = response.json()
print(patientJSON)

patientId = None


if ((patientJSON['total']> 0) and (len(patientJSON['entry'])>0)):
    patientId = patientJSON['entry'][0]['resource']['id']
    print()
    print("Patient = " + patientId)

Now lets find diagnostic reports for this patient.

This code:

- Checks whether a patient has already been found
- If so, asks the FHIR server for that patient’s Diagnostic Reports
  - Reads the response
  - Looks at the first diagnostic report
  - Extracts the organisation that performed the report
  - Prints that organisation’s ID

In short:
👉 “Given a patient, find which organisation produced their diagnostic report.”

In [ ]:
organisationId = None

if patientId != None:
    api_url = server + "DiagnosticReport?patient="+"Patient/"+patientId
    response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
    diagnosticReportsJSON = response.json()
    print(diagnosticReportsJSON)
    if (len(diagnosticReportsJSON['entry'])>0):
        organisationId = diagnosticReportsJSON['entry'][0]['resource']['performer'][0]['reference'].replace('Organization/', '')
        print()
        print("Organization = " + organisationId)


Can also return details about the organisation that created this.

This data is sourced from [NHS England Organisation Data Terminology - FHIR API](https://digital.nhs.uk/developer/api-catalogue/organisation-data-terminology). Note: the model used here is simplified.

In [ ]:
if organisationId != None:
    api_url = server + "Organization/"+organisationId
    print(api_url)
    response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
    organisationJSON = response.json()
    print(organisationJSON)

Example using python to interpret the json and also using [fhirclient](https://github.com/smart-on-fhir/client-py)

In [ ]:
import fhirclient.models.diagnosticreport as dr
import pandas as pd

api_url = server + "DiagnosticReport?_count=50"
response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
response1JSON = response.json()

print(response1JSON['total'])

reports = []
for entry in response1JSON['entry']:
    #print(entry['resource'])
    print(entry['resource']['resourceType'], entry['resource']['issued'] )
    report = dr.DiagnosticReport(entry['resource'])
    reports.append(report)
    for coding in report.code.coding:
        print(coding.code)

We are aiming at display graphs based on the order and the report.

We can include the order which in FHIR is held in ServiceRequest by including this in the query.

High-level purpose

This code:

- Queries a FHIR server for DiagnosticReports
- Uses _include to also fetch related ServiceRequests and Specimens
- Handles paged results
- Converts raw FHIR JSON into FHIR client model objects
- Collects everything into lists for later processing

In short:
👉 “Fetch diagnostic reports and all their directly related clinical context in one sweep.”

In [ ]:
import fhirclient.models.servicerequest as sr
import fhirclient.models.specimen as sp

serviceRequests = []
diagnosticReports = []
specimens = []

api_url = server + "DiagnosticReport?_include=DiagnosticReport:based-on&_include=DiagnosticReport:specimen&_lastUpdated=gt2025-12-01"


while True:
    response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
    responseInclude = response.json()

    #print(responseInclude)
    print(responseInclude['total'])
    entry = responseInclude['entry']
    print(len(entry))
    if len(entry) == 0:
        break
    for entry in responseInclude['entry']:

        if entry['resource']['resourceType'] == 'DiagnosticReport':
            report = dr.DiagnosticReport(entry['resource'])
            diagnosticReports.append(report)
        if entry['resource']['resourceType'] == 'ServiceRequest':
            request = sr.ServiceRequest(entry['resource'])
            serviceRequests.append(request)
        if entry['resource']['resourceType'] == 'Specimen':
            specimen = sp.Specimen(entry['resource'])
            specimens.append(specimen)

    print("ServiceRequest = " + str(len(serviceRequests)))
    print("DiagnosticReport = " + str(len(diagnosticReports)))
    print("Specimen = " + str(len(specimens)))

    found = False
    for link in responseInclude['link']:
        if link['relation'] == 'next':
            api_url = link['url']
            found = True
            print(api_url)
    if found == False:
        break

Process the diaganostic reports

This code defines a small set of helper functions used to extract specific values from FHIR model objects, primarily for display and reporting purposes. The functions simplify access to commonly needed fields that are often represented in FHIR as lists or nested structures.

In [ ]:
import fhirclient.models.meta as meta
from dateutil import parser

def performer(my_list):
    performer = ""
    if my_list != None:
        for item in my_list:
            performer = item.display
    return performer

def performerCode(my_list):
    performer = None
    if my_list != None:
        for item in my_list:
            performer = item.identifier.value
    return performer

def codeCode(concept):
    code = ""
    for coding in concept.coding:
        if coding.system == "https://fhir.nwgenomics.nhs.uk/CodeSystem/IGEAP":
            code = coding.code

    return code
def codeDisplay(concept):
    code = ""
    for coding in concept.coding:
        if coding.system == "https://fhir.nwgenomics.nhs.uk/CodeSystem/IGEAP":
            code = coding.display

    return code

def issued(issued):
    if issued == None:
        return None
    return parser.parse(issued.isostring)

def serviceRequest(my_list):
    sr = None
    if my_list != None:
        for item in my_list:
            if item.reference != None:
                sr = item.reference.replace('ServiceRequest/', '')
    return sr
def specimen(my_list):
    sr = None
    if my_list != None:
        for item in my_list:
            if item.reference != None:
                sr = item.reference.replace('Specimen/', '')
    return sr
def lastUpdated(meta : meta.Meta):
    if meta == None:
        return None
    return parser.parse(meta.lastUpdated.isostring)

print(len(diagnosticReports))
dfDR = pd.DataFrame([vars(s) for s in diagnosticReports])

dfDR['performerDisplay'] = dfDR['performer'].apply(performer)
dfDR['performerCode'] = dfDR['performer'].apply(performerCode)
dfDR['codingCode'] = dfDR['code'].apply(codeCode)
dfDR['codingDisplay'] = dfDR['code'].apply(codeDisplay)
dfDR['ReportLastUpdatedDate'] = dfDR['meta'].apply(lastUpdated)
dfDR['ReportIssuedDate'] = dfDR['issued'].apply(issued)
dfDR['ReportEffectiveDate'] = dfDR['effectiveDateTime'].apply(issued)
dfDR['serviceRequestId'] = dfDR['basedOn'].apply(serviceRequest)
dfDR['specimenId'] = dfDR['specimen'].apply(specimen)

dfDiagnosticReport = dfDR[['id','performerDisplay','performerCode','codingCode', 'codingDisplay', 'ReportLastUpdatedDate','ReportIssuedDate', 'ReportEffectiveDate', 'serviceRequestId','specimenId']]
dfDiagnosticReport

In [ ]:
print(len(specimens))
dfSP = pd.DataFrame([vars(s) for s in specimens])
dfSP['SpecimenReceivedDate'] = dfSP['receivedTime'].apply(issued)


dfSpecimen = dfSP[['id','SpecimenReceivedDate']]
dfSpecimen

Clean up requests

In [ ]:
print(len(serviceRequests))

def requester(item):
    performer = None
    if item != None:
        performer = item.display
    return performer

def requesterCode(item):
    performer = None
    if item != None:
        performer = item.identifier.value
    return performer

def CICode(my_list):
    code = None
    if my_list != None:
        for concept in my_list:
            for coding in concept.coding:
                code = coding.code
    return code
def CIDisplay(my_list):
    code = None
    if my_list != None:
        for concept in my_list:
            for coding in concept.coding:
                code = coding.display
    return code

dfSR = pd.DataFrame([vars(s) for s in serviceRequests])
dfSR['requesterDisplay'] = dfSR['requester'].apply(requester)
dfSR['requesterCode'] = dfSR['requester'].apply(requesterCode)
dfSR['OrderDate'] = dfSR['authoredOn'].apply(issued)
dfSR['CICode'] = dfSR['reasonCode'].apply(CICode)
dfSR['CIDisplay'] = dfSR['reasonCode'].apply(CIDisplay)

dfServiceRequest = dfSR[['id','requesterDisplay','requesterCode','OrderDate','CICode','CIDisplay']]
dfServiceRequest

Join both dataframes into a single result.

In [ ]:
# ... existing code ...
df = pd.merge(
    dfDiagnosticReport,
    dfServiceRequest,
    left_on='serviceRequestId',
    right_on='id',
    how="left",
    indicator=True,
    suffixes=('_dr', '_sr')
)
df = df.drop(columns=['_merge'])
df = pd.merge(
    df,
    dfSpecimen,
    left_on='specimenId',
    right_on='id',
    how="left",
    indicator=True,
    suffixes=('_dr', '_sp')
)


df['ReportIssuedDate'] = pd.to_datetime(df['ReportIssuedDate'], utc=True).dt.tz_localize(None)
df['OrderDate'] = pd.to_datetime(df['OrderDate'], utc=True).dt.tz_localize(None)
df['ReportLastUpdatedDate'] = pd.to_datetime(df['ReportLastUpdatedDate'], utc=True).dt.tz_localize(None)

df['requestedDuration'] = (df['ReportIssuedDate'] - df['OrderDate']).dt.days

df['releaseDuration'] = (df['ReportLastUpdatedDate'] - df['ReportIssuedDate']).dt.days
df['releaseDurationMin'] = (df['ReportLastUpdatedDate'] - df['ReportIssuedDate']).dt.total_seconds() / 60
df['testingDuration'] = (df['ReportIssuedDate'] - df['SpecimenReceivedDate']).dt.days
df['OrderToSpecimenReceivedDuration'] = (df['SpecimenReceivedDate'] - df['OrderDate']).dt.days

#df['releaseDuration'] = df['releaseDuration'].fillna(-1)
#df['requestedDuration'] = df['requestedDuration'].fillna(-1)
df['OrderToSpecimenReceivedDuration'] = df['OrderToSpecimenReceivedDuration'].fillna(-1)

df

In [ ]:
import plotly.graph_objects as go


fig = go.Figure()
dfTotals = df.requesterDisplay.value_counts().reset_index(name='counts')
fig.add_trace(go.Bar(x=dfTotals['requesterDisplay'], y=dfTotals['counts']))
fig.show()

fig2 = go.Figure()
dfTotals2 = df.codingDisplay.value_counts().reset_index(name='counts')
fig2.add_trace(go.Bar(x=dfTotals2['codingDisplay'], y=dfTotals['counts']))
fig2.show()



In [ ]:
import plotly.express as px

df1 = df.groupby(['requesterCode', 'codingCode']).size().reset_index(name='count')

fig = px.bar(df1, x="requesterCode", y="count", color="codingCode")

fig.show()
#px.bar(df1, x="requesterCode", y="count", color="codingCode", barmode="group")

In [ ]:
dateRangeStart = pd.Timestamp(2025,11,1)

dfS = df[['SpecimenReceivedDate', 'testingDuration', 'codingCode']]
dfS = dfS.dropna(subset=['SpecimenReceivedDate'])
dfS = dfS[(dfS['SpecimenReceivedDate'] > dateRangeStart)]
dfS = dfS.groupby(['SpecimenReceivedDate', 'testingDuration', 'codingCode']).size().reset_index(name='counts')
dfS


In [ ]:
# Now the scatter plot will work as expected
fig = px.scatter(
    dfS,
    x="SpecimenReceivedDate",
    y="testingDuration",
    color="codingCode",
    size = "counts",
    title="Testing Time from Specimen Collection to Report Release"
)

fig.show()

In [ ]:
dti = pd.date_range("2025-10-01", periods=100, freq="d").to_frame(index=False, name="date")
dti['date'] = dti['date'].dt.date

df['ReportSentDT'] = df['ReportLastUpdatedDate'].dt.date
df['ReportIssuedDT'] = df['ReportIssuedDate'].dt.date
df['OrderAuthoredDT'] = df['OrderDate'].dt.date
df['SpecimenReceivedDT'] = df['SpecimenReceivedDate'].dt.date

df = df[(df['SpecimenReceivedDate'] > dateRangeStart)]

dfA = df.groupby(['ReportSentDT']).size().reset_index(name='ReportSent')
dfB = df.groupby(['ReportIssuedDT']).size().reset_index(name='ReportIssued')
dfC = df.groupby(['OrderAuthoredDT']).size().reset_index(name='OrderAuthored')
dfD = df.groupby(['SpecimenReceivedDT']).size().reset_index(name='SpecimenReceived')

df_merged = pd.merge(dti, dfA, left_on='date', right_on='ReportSentDT', how='left')
df_merged = pd.merge(df_merged, dfB, left_on='date', right_on='ReportIssuedDT', how='left')
df_merged = pd.merge(df_merged, dfC, left_on='date', right_on='OrderAuthoredDT', how='left')
df_merged = pd.merge(df_merged, dfD, left_on='date', right_on='SpecimenReceivedDT', how='left')
df_merged.drop(columns=['ReportSentDT', 'ReportIssuedDT', 'OrderAuthoredDT','SpecimenReceivedDT'], inplace=True)


dfmelt = df_merged.melt(id_vars=["date"],
        var_name="DateType",
        value_name="Count")

#dfmelt = dfmelt.groupby(['date', 'Value', 'DateType']).size().reset_index(name='counts')

fig = px.scatter(dfmelt,x="date", y="Count", color="DateType",  title="Last Updated")
fig.show()

fig = px.line(dfmelt,x="date", y="Count", color="DateType",  title="Last Updated")
fig.show()

dfmelt





In [ ]:
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc

# Initialize the Dash app (no longer need JupyterDash)
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

@app.callback(
    Output('reportsByRequester', 'figure')
)
def reportsByRequester():
    dfReports.sort_values('count', inplace=True, ascending=False)

    fig = px.bar(dfReports, x="requesterCode", y="count", color="codingCode", title="Report Sent by NHS Trust")

    return fig

@app.callback(
    Output('reportsByTestCode', 'figure')
)
def reportsByTestCode():
    figTC = px.bar(dfReports, x="codingCode", y="count", color="requesterCode", title="Report Sent by Test Code")
    return figTC

@app.callback(
    Output('durations', 'figure')
)
def durations():
    dfS = df[['SpecimenReceivedDate', 'testingDuration', 'codingCode']]
    dfS = dfS.dropna(subset=['SpecimenReceivedDate'])
    dfS = dfS.groupby(['SpecimenReceivedDate', 'testingDuration', 'codingCode']).size().reset_index(name='counts')

    # Now the scatter plot will work as expected
    figD = px.bar(
        dfS,
        x="testingDuration",
        y="counts",
        color="codingCode",
        title="Testing Time from Specimen Collection to Report Release"
    )

    return figD

@app.callback(
    Output('specimens', 'figure')
)
def specimensByNHS():
    dfS = df[['OrderDate', 'OrderToSpecimenReceivedDuration', 'requesterCode']]
    dfS = dfS.dropna(subset=['OrderDate'])
    dfS = dfS.groupby(['OrderDate', 'OrderToSpecimenReceivedDuration', 'requesterCode']).size().reset_index(name='counts')

    # Now the scatter plot will work as expected
    figD = px.bar(
        dfS,
        x="OrderToSpecimenReceivedDuration",
        y="counts",
        color="requesterCode",
        title="Time from Order from NHS Trust to Specimen Received"
    )

    return figD

@app.callback(
    Output('specimensByCode', 'figure')
)
def specimensByCode():
    dfS = df[['OrderDate', 'OrderToSpecimenReceivedDuration', 'codingCode']]
    dfS = dfS.dropna(subset=['OrderDate'])
    dfS = dfS.groupby(['OrderDate', 'OrderToSpecimenReceivedDuration', 'codingCode']).size().reset_index(name='counts')

    # Now the scatter plot will work as expected
    figD = px.bar(
        dfS,
        x="OrderToSpecimenReceivedDuration",
        y="counts",
        color="codingCode",
        title="Time from Order to Specimen Received by Test Code"
    )

    return figD

@app.callback(
    Output('release', 'figure')
)
def release():
    dfS = df[['ReportIssuedDate', 'releaseDuration', 'codingCode']]
    dfS = dfS.dropna(subset=['ReportIssuedDate'])
    dfS = dfS.groupby(['ReportIssuedDate', 'releaseDuration', 'codingCode']).size().reset_index(name='counts')
    # Now the scatter plot will work as expected
    figE = px.bar(
        dfS,
        x="releaseDuration",
        y="counts",
        color="codingCode",
        title="Time from Report Release to Report Sent"
    )
    return figE

@app.callback(
    Output('releaseLine', 'figure')
)
def releaseLine():
    dfS = df[['ReportLastUpdatedDate', 'releaseDurationMin']]
    dfS = dfS.dropna(subset=['ReportLastUpdatedDate'])
    dfS = dfS.sort_values('releaseDurationMin')
    # Now the scatter plot will work as expected
    figE = px.scatter(
        dfS,
        x="ReportLastUpdatedDate",
        y="releaseDurationMin",
        title="Release Duration over time"
    )
    return figE

@app.callback(
    Output('lines', 'figure')
)
def lines():

    figL = px.bar(dfmelt,x="date", y="Count", color="DateType",  title="Timeline overview for sent reports")
    return figL



dfReports = df.groupby(['requesterCode', 'codingCode']).size().reset_index(name='count')

fig1 = reportsByRequester()
fig3 = reportsByTestCode()
fig2 = durations()
fig4 = lines()
fig51 = specimensByNHS()
fig52 = specimensByCode()
fig6 = release()
figRL = releaseLine()



app.layout = html.Div([
    html.H1("NW Genomics HIE REST Dashboard"),
    dbc.Row([
        dbc.Col([
            dcc.Graph(id='lines', figure=fig4),
        ]),
    ]),
    dbc.Row([
        dbc.Col([
            dcc.Graph(id='releaseLine', figure=figRL),
        ]),
    ]),
    dbc.Row([
        dbc.Col([
            dcc.Graph(id='specimens', figure=fig51),
        ]),
        dbc.Col([
            dcc.Graph(id='specimensByCode', figure=fig52),
        ]),
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='durations', figure=fig2)
        ]),
        dbc.Col([
            dcc.Graph(id='release', figure=fig6),
        ])
    ]),
    dbc.Row([
        dbc.Col([
            dcc.Graph(id='reportsByRequester', figure=fig1),
        ]),
        dbc.Col([
            dcc.Graph(id='reportByTestCode', figure=fig3),
        ])

    ])
])


if __name__ == '__main__':
    # Use app.run with jupyter_mode instead of JupyterDash's run_server
    app.run(jupyter_mode='inline', port=8051)